# PSQ Customer Base v8 — with RICOS Analysis

**Version:** v8.0  
**Purpose:** Build PSQ customer base (Way4 + PASS) and analyze RICOS match coverage.  
**Outputs:**
- `way4_base_v8` — Way4 customer base (8,039 rows, 25 columns)
- `pass_base_v8` — PASS customer base (10,328 rows, 25 columns)
- `psq_customer_base_v8` — Combined base (18,367 rows, 25 columns)
- `ricos_psq_merchants` — RICOS data filtered to PSQ only (P_M, PSQ flag)
- `ricos_risk_lookup` — RICOS risk scores for PSQ merchants
- `psq_match_summary` — Match rates: WAY4 71.4%, PASS 35.3%
- `psq_with_ricos_flag` — Full PSQ base + in_ricos_flag (Y/N)

---

## Source Tables (13 total)

| System | Table | Purpose |
|--------|-------|---------|
| Way4 | prod.bi_dwh_wlbe_way4.factpayment | Transactions |
| Way4 | prod.bi_dwh_wlbe_way4.dimcompany | Company master |
| Way4 | prod.bi_dwh_wlbe_way4.dimcontract | Contract + MCC |
| Way4 | prod.bi_ods_wlbe_way4.PSQ_Merchant_Data | ODS identifiers |
| PASS | prod.bi_dwh_worldlinenl.facttransaction_dama | Transactions |
| PASS | prod.bi_dwh_worldlinenl.dimserviceaccount_dama | Service accounts |
| PASS | prod.bi_dwh_worldlinenl.dimcompany_dama | Company master |
| PASS | prod.bi_dwh_worldlinenl.dimcontract_dama | Contract categories |
| PASS | prod.bi_ods_worldlinenl.dama_tk_par | ODS partner (LEGALFORM, KvK) |
| PASS | prod.bi_ods_worldlinenl.dama_tk_cod | Code reference table |
| RICOS | prod.bronze_ricos_prod.gwgkunde4400 | Merchant master |
| RICOS | prod.bronze_ricos_prod.presult4400 | Risk/screening results |
| RICOS | prod.bronze_ricos_prod.tbbo4400 | UBO/SI relationships |

## Column Changes vs v8.3
- 25 columns (was 24): added `sourcecoderef` (PASS = dama_tk_par.SOURCECODEREF, Way4 = NULL)

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime

# ══════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════

REPORTING_DATE = "2026-05-13"   # Fixed reporting date for auditability

# Derived date expressions
reporting_date = F.to_date(F.lit(REPORTING_DATE))
cutoff_12m = F.add_months(reporting_date, -12)   # v8: 12 months (was 13)
cutoff_6m = F.add_months(reporting_date, -6)

# VAT placeholder values (known invalid entries)
INVALID_VAT_VALUES = ["", "1", "U", "NL", "UNKNOWN", "UNKN", "NA", "N/A"]

print(f"REPORTING_DATE:  {REPORTING_DATE}")
print(f"Version:         v8.0")
print(f"Activity cutoff: 12 months")

In [ ]:
# ══════════════════════════════════════════════════════════════
# LOAD ALL SOURCE TABLES (13 tables)
# ══════════════════════════════════════════════════════════════
# Way4 (4) + PASS (6) + RICOS (3)
# All read-only — no writes to source systems.
# ══════════════════════════════════════════════════════════════

# --- WAY4 (DWH silver + ODS) ---
way4_fp       = spark.table("prod.bi_dwh_wlbe_way4.factpayment")          # Source: Transactions (Amount, DatePK, ServiceTypeBK, ContractBK, CurrencyBK)
way4_dimcom   = spark.table("prod.bi_dwh_wlbe_way4.dimcompany")           # Source: Company master (LegalName, CountryBK, VintageDate)
way4_dimcon   = spark.table("prod.bi_dwh_wlbe_way4.dimcontract")          # Source: Contract master (MCC, ContractBK)
way4_psq_ods  = spark.table("prod.bi_ods_wlbe_way4.PSQ_Merchant_Data")    # Source: ODS (CONTRACT_ID, VAT, KvK, TERMINATION_DATE, CompanyBK, ContractBK)

# --- PASS / WLNL (DWH silver + ODS) ---
pass_ft       = spark.table("prod.bi_dwh_worldlinenl.facttransaction_dama")      # Source: Transactions (AmountValue, DatePK, ClearingState=41)
pass_dim_sa   = spark.table("prod.bi_dwh_worldlinenl.dimserviceaccount_dama")    # Source: Service accounts (ServiceAccountBK→CompanyBK, MCCode)
pass_dim_com  = spark.table("prod.bi_dwh_worldlinenl.dimcompany_dama")           # Source: Company master (CompanyCode, CompanyName, VATNUMBER, CountryBK, SourceStatusDescription)
pass_dim_ct   = spark.table("prod.bi_dwh_worldlinenl.dimcontract_dama")          # Source: Contract (ContractCategory, ContractCategoryDescription, ContractStatus)
pass_ods_par  = spark.table("prod.bi_ods_worldlinenl.dama_tk_par")               # Source: ODS partner (LEGALFORM, REGISTRY=KvK, CREATEDDATE=vintage, SOURCECODEREF)
pass_ods_cod  = spark.table("prod.bi_ods_worldlinenl.dama_tk_cod")               # Source: Code reference (CODETYPE, CODEVAL, CODEDESCRIPTION)

# --- RICOS (Bronze layer) ---
ricos_gwg     = spark.table("prod.bronze_ricos_prod.gwgkunde4400")               # Source: Merchant master (KUNDNR, KD_SPH_03, BRANCHE, H_LAND, NACHNAME, KD_SPH_25)
ricos_presult = spark.table("prod.bronze_ricos_prod.presult4400")                # Source: Risk/screening results (RISIKO, RISK_ORIG, RISK_MAN, STATUS, PSTATUS)
ricos_tbbo    = spark.table("prod.bronze_ricos_prod.tbbo4400")                   # Source: UBO/SI relationships (REL_TYPE, REL_KUNDNR, KUNDNR)

print("Source tables loaded: 13 (Way4 4 + PASS 6 + RICOS 3)")
print("  Way4:  factpayment, dimcompany, dimcontract, PSQ_Merchant_Data")
print("  PASS:  facttransaction_dama, dimserviceaccount_dama, dimcompany_dama, dimcontract_dama, dama_tk_par, dama_tk_cod")
print("  RICOS: gwgkunde4400, presult4400, tbbo4400")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATA EXPLORATION: OFFICIAL CODE REFERENCE VALUES
# ══════════════════════════════════════════════════════════════
# Source: prod.bi_ods_worldlinenl.dama_tk_cod
# Official lookup table for all PASS code values.
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  PASS CODE REFERENCE VALUES (from dama_tk_cod)")
print("═" * 70)

# 1. Contract Status (ContStatus)
print("\n  1. CONTRACT STATUS (CODETYPE = 'ContStatus')")
print("     Used for: active_contracts filter (ContractStatus = 1)")
display(
    pass_ods_cod
    .filter(F.col("CODETYPE") == "ContStatus")
    .select("CODEVAL", "CODEMNEMONIC", "CODEDESCRIPTION")
    .orderBy("CODEVAL")
)

# 2. Contract Category Group
print("\n  2. CONTRACT CATEGORY GROUP (CODETYPE = 'ContractCategoryGroup')")
print("     Used for: type_of_activity and contracting_method derivation")
display(
    pass_ods_cod
    .filter(F.col("CODETYPE") == "ContractCategoryGroup")
    .select("CODEVAL", "CODEMNEMONIC", "CODEDESCRIPTION")
    .orderBy("CODEVAL")
)

# 3. Actual ContractCategory values in dimcontract_dama
print("\n  3. ACTUAL CONTRACT CATEGORY VALUES (from dimcontract_dama)")
print("     These are the real values in the data (1, 8, 9, 10):")
display(
    pass_dim_ct
    .groupBy("ContractCategory", "ContractCategoryDescription")
    .agg(F.count("*").alias("contract_count"))
    .orderBy("ContractCategory")
)

# 4. Account/Partner Status
print("\n  4. ACCOUNT STATUS (CODETYPE = 'AccStatus')")
print("     Maps to: dimcompany_dama.SourceStatus")
display(
    pass_ods_cod
    .filter(F.col("CODETYPE") == "AccStatus")
    .select("CODEVAL", "CODEMNEMONIC", "CODEDESCRIPTION")
    .orderBy("CODEVAL")
)

# 5. Way4 ServiceType distribution (confirm all ACQUIRING)
print("\n  5. WAY4 SERVICE TYPE (from factpayment.ServiceTypeBK)")
print("     Confirms Way4 = pure POS acquiring (no ecommerce)")
display(
    way4_fp
    .groupBy("ServiceTypeBK")
    .agg(F.count("*").alias("trx_count"))
    .orderBy(F.desc("trx_count"))
)

print("\n  Reference values loaded. Used to derive type_of_activity, contracting_method, contract_status.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# WAY4 CUSTOMER BASE v8.1
# ══════════════════════════════════════════════════════════════
# Grain: 1 row per CONTRACT_ID
# Output: 25 columns (24 original + sourcecoderef)
#
# Tables used (aliases in join):
#   psq = prod.bi_ods_wlbe_way4.PSQ_Merchant_Data     ← anchor table (CONTRACT_ID, CompanyBK, ContractBK)
#   com = prod.bi_dwh_wlbe_way4.dimcompany            ← entity master (LegalName, CountryBK, VintageDate)
#   dc  = prod.bi_dwh_wlbe_way4.dimcontract           ← contract details (MCC)
#   trx = prod.bi_dwh_wlbe_way4.factpayment (12m agg) ← 12m metrics (Amount, DatePK, ServiceTypeBK)
#   lt  = prod.bi_dwh_wlbe_way4.factpayment (all-time) ← last ever transaction DatePK
# ══════════════════════════════════════════════════════════════

from dateutil.relativedelta import relativedelta

_rpt_dt = datetime.strptime(REPORTING_DATE, "%Y-%m-%d")
_cutoff_dt = _rpt_dt - relativedelta(months=12)
cutoff_datepk_12m = int(_cutoff_dt.strftime("%Y%m%d"))

# Step 1: 12-month transaction metrics per ContractBK
# Source: prod.bi_dwh_wlbe_way4.factpayment
way4_trx_agg = (
    way4_fp
    .filter(F.col("DatePK") >= cutoff_datepk_12m)
    .filter(F.col("CurrencyBK") != "-1")  # Exclude 72 sentinel currency rows
    .groupBy("ContractBK")
    .agg(
        F.count("*").alias("trx_count_12m"),
        F.sum("Amount").cast("decimal(20,2)").alias("total_msv_12m_eur"),
        F.max("Amount").cast("decimal(20,2)").alias("max_trx_amount_12m_eur"),
        F.min("DatePK").alias("first_trx_datepk_12m"),
        F.max("DatePK").alias("recent_trx_datepk_12m"),
        F.collect_set("ServiceTypeBK").alias("service_types"),
    )
)

# Step 2: All-time last transaction date
# Source: prod.bi_dwh_wlbe_way4.factpayment (full history)
way4_last_trx = (
    way4_fp
    .filter(F.col("CurrencyBK") != "-1")
    .groupBy("ContractBK")
    .agg(F.max("DatePK").alias("last_trx_datepk"))
)

# Step 3: Assemble Way4 base (25 columns)
way4_base_v8 = (
    way4_psq_ods.alias("psq")
    .join(way4_dimcom.alias("com"), F.col("psq.CompanyBK") == F.col("com.CompanyBK"), "left")
    .join(way4_dimcon.alias("dc"), F.col("psq.ContractBK") == F.col("dc.ContractBK"), "left")
    .join(way4_trx_agg.alias("trx"), F.col("psq.ContractBK") == F.col("trx.ContractBK"), "left")
    .join(way4_last_trx.alias("lt"), F.col("psq.ContractBK") == F.col("lt.ContractBK"), "left")
    .select(
        F.lit("WAY4").alias("source"),
        # Source: constructed from psq.CONTRACT_ID
        F.concat(F.lit("WAY4:"), F.col("psq.CONTRACT_ID")).alias("psq_entity_key"),
        # Source: prod.bi_ods_wlbe_way4.PSQ_Merchant_Data.CONTRACT_ID
        F.col("psq.CONTRACT_ID").alias("id"),
        # Source: NULL for Way4 (no SOURCECODEREF equivalent)
        F.lit(None).cast("string").alias("sourcecoderef"),
        # Source: prod.bi_dwh_wlbe_way4.dimcompany.LegalName
        F.col("com.LegalName").alias("name"),
        # Source: prod.bi_ods_wlbe_way4.PSQ_Merchant_Data.LegalIdentifier_vat_numberBK (strip 'WAY4;' prefix)
        F.when(
            F.col("psq.LegalIdentifier_vat_numberBK").isin("-1", ""), F.lit(None)
        ).when(
            F.col("psq.LegalIdentifier_vat_numberBK").startswith("WAY4;"),
            F.regexp_replace(F.col("psq.LegalIdentifier_vat_numberBK"), "^WAY4;", "")
        ).otherwise(F.col("psq.LegalIdentifier_vat_numberBK")).alias("vat_number"),
        # Source: prod.bi_ods_wlbe_way4.PSQ_Merchant_Data.LegalIdentifier_chamber_of_commerce_numberBK (strip 'WAY4;' prefix)
        F.when(
            F.col("psq.LegalIdentifier_chamber_of_commerce_numberBK").isin("-1", ""), F.lit(None)
        ).when(
            F.col("psq.LegalIdentifier_chamber_of_commerce_numberBK").startswith("WAY4;"),
            F.regexp_replace(F.col("psq.LegalIdentifier_chamber_of_commerce_numberBK"), "^WAY4;", "")
        ).otherwise(F.col("psq.LegalIdentifier_chamber_of_commerce_numberBK")).alias("kvk_number"),
        # Source: prod.bi_dwh_wlbe_way4.dimcompany.CountryBK
        F.col("com.CountryBK").alias("country"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.DatePK (all-time max)
        F.to_date(F.col("lt.last_trx_datepk").cast("string"), "yyyyMMdd").alias("last_trx_date"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.DatePK (12m min)
        F.to_date(F.col("trx.first_trx_datepk_12m").cast("string"), "yyyyMMdd").alias("first_trx_date_12m"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.DatePK (12m max)
        F.to_date(F.col("trx.recent_trx_datepk_12m").cast("string"), "yyyyMMdd").alias("recent_trx_date_12m"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment count(*) over 12m
        F.coalesce(F.col("trx.trx_count_12m"), F.lit(0)).alias("trx_count_12m"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.Amount (sum, 12m, EUR)
        F.coalesce(F.col("trx.total_msv_12m_eur"), F.lit(0).cast("decimal(20,2)")).alias("total_msv_12m_eur"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.Amount (max single trx, 12m)
        F.col("trx.max_trx_amount_12m_eur"),
        # Derived: "Active in last 12 months" if trx_count_12m > 0
        F.when(
            F.col("trx.trx_count_12m").isNotNull() & (F.col("trx.trx_count_12m") > 0),
            "Active in last 12 months"
        ).otherwise("No transactions in last 12 months").alias("activity_detail_status"),
        # Derived: binary Active/Inactive
        F.when(
            F.col("trx.trx_count_12m").isNotNull() & (F.col("trx.trx_count_12m") > 0), "Active"
        ).otherwise("Inactive").alias("merchant_activity_status"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.ServiceTypeBK (100% WAY4;ACQUIRING → always CP)
        F.lit("CP").alias("type_of_activity"),
        # Source: prod.bi_dwh_wlbe_way4.factpayment.ServiceTypeBK (same logic as type_of_activity)
        F.lit("F2F").alias("contracting_method"),
        # Source: NULL (Way4 has no legal_form in any native table, confirmed by table scan)
        F.lit(None).cast("string").alias("legal_form"),
        # Source: prod.bi_dwh_wlbe_way4.dimcontract.MCC
        F.col("dc.MCC").cast("string").alias("mcc"),
        # Source: prod.bi_dwh_wlbe_way4.dimcompany.VintageDate
        F.col("com.VintageDate").alias("vintage_date"),
        # Source: prod.bi_ods_wlbe_way4.PSQ_Merchant_Data.CONTRACT_TERMINATION_DATE (NULL=Active, populated=Terminated)
        F.when(
            F.col("psq.CONTRACT_TERMINATION_DATE").isNull(), "Active"
        ).otherwise("Terminated").alias("contract_status"),
        # Source: NULL (Way4 = 1 contract per merchant by design)
        F.lit(None).cast("int").alias("active_contracts"),
        # Source: NULL (Way4 has no contract categories)
        F.lit(None).cast("string").alias("contract_category_descriptions"),
        # Derived: flags missing fields
        F.concat_ws("; ",
            F.when(F.col("psq.LegalIdentifier_vat_numberBK").isin("-1", "", None), F.lit("missing_vat")),
            F.when(F.col("psq.LegalIdentifier_chamber_of_commerce_numberBK").isin("-1", "", None), F.lit("missing_kvk")),
            F.when(F.col("dc.MCC").isNull(), F.lit("missing_mcc")),
            F.when(F.col("com.VintageDate").isNull(), F.lit("missing_vintage")),
        ).alias("data_quality_notes"),
    )
)

way4_count = way4_base_v8.count()
print(f"Way4 Customer Base v8.1: {way4_count:,} rows")
print(f"Columns ({len(way4_base_v8.columns)}): {way4_base_v8.columns}")

In [ ]:
display(way4_base_v8)

In [ ]:
# ══════════════════════════════════════════════════════════════
# PASS (WLNL) CUSTOMER BASE v8.1
# ══════════════════════════════════════════════════════════════
# Grain: 1 row per CompanyCode (= PARTNERID, validated 100%)
# Output: 25 columns (includes sourcecoderef from dama_tk_par.SOURCECODEREF)
#
# Tables used (aliases in join):
#   com   = prod.bi_dwh_worldlinenl.dimcompany_dama          ← anchor (CompanyCode, CompanyName, CountryBK, VATNUMBER)
#   trx   = prod.bi_dwh_worldlinenl.facttransaction_dama     ← 12m metrics (AmountValue, DatePK, ClearingState=41)
#   s     = prod.bi_dwh_worldlinenl.dimserviceaccount_dama   ← link trx→company (ServiceAccountBK→CompanyBK), MCCode
#   lt    = prod.bi_dwh_worldlinenl.facttransaction_dama     ← all-time max DatePK
#   ac    = prod.bi_ods_worldlinenl.dama_tk_con              ← active contract count
#   cod   = prod.bi_ods_worldlinenl.dama_tk_cod              ← code reference (CODETYPE=ContStatus, CODEVAL=1)
#   ct    = prod.bi_dwh_worldlinenl.dimcontract_dama         ← ContractCategory (1=F2F, 8=ecom, 9=F2F DCC, 10=ecom DCC)
#   mcc_t = prod.bi_dwh_worldlinenl.dimserviceaccount_dama   ← MCCode (min per company)
#   ods   = prod.bi_ods_worldlinenl.dama_tk_par              ← LEGALFORM, REGISTRY (=KvK), CREATEDDATE (=vintage), SOURCECODEREF
# ══════════════════════════════════════════════════════════════

# Constants: F2F and NF2F contract categories (data-driven)
F2F_CATEGORIES = [1, 9]      # face to face, face to face DCC
NF2F_CATEGORIES = [8, 10]    # secure e-commerce, secure e-commerce DCC
SENTINEL_CODES = ["-1", "-2", "-3", "-4", "-5"]

# SCD filter: current records only (SCDEndDate IS NULL)
pass_dim_sa_current = pass_dim_sa.filter(F.col("SCDEndDate").isNull())
pass_dim_com_current = pass_dim_com.filter(F.col("SCDEndDate").isNull())
pass_ods_par_current = pass_ods_par.filter(F.col("SCDEndDate").isNull())

# Step 1: 12-month transaction metrics per CompanyBK
# Source: prod.bi_dwh_worldlinenl.facttransaction_dama.AmountValue (EUR, settled only ClearingState=41)
pass_trx_12m = (
    pass_ft.filter(F.col("ClearingState") == 41)
    .filter(F.col("DatePK") >= cutoff_datepk_12m)
    .alias("f")
    .join(
        pass_dim_sa_current.select("ServiceAccountBK", "CompanyBK").alias("s"),
        F.col("f.PartnerServiceAccountBK") == F.col("s.ServiceAccountBK"), "inner"
    )
    .groupBy(F.col("s.CompanyBK"))
    .agg(
        F.count(F.col("f.PartnerServiceAccountBK")).alias("trx_count_12m"),
        F.sum("f.AmountValue").cast("decimal(20,2)").alias("total_msv_12m_eur"),
        F.max("f.AmountValue").cast("decimal(20,2)").alias("max_trx_amount_12m_eur"),
        F.min("f.DatePK").alias("first_trx_datepk_12m"),
        F.max("f.DatePK").alias("recent_trx_datepk_12m"),
    )
)

# Step 2: All-time last transaction date
# Source: prod.bi_dwh_worldlinenl.facttransaction_dama (full history, ClearingState=41)
pass_last_trx = (
    pass_ft.filter(F.col("ClearingState") == 41).alias("f")
    .join(
        pass_dim_sa_current.select("ServiceAccountBK", "CompanyBK").alias("s"),
        F.col("f.PartnerServiceAccountBK") == F.col("s.ServiceAccountBK"), "inner"
    )
    .groupBy(F.col("s.CompanyBK"))
    .agg(F.max("f.DatePK").alias("last_trx_datepk"))
)

# Step 3: Active contracts count (Vamshi ODS logic)
# Source: prod.bi_ods_worldlinenl.dama_tk_con + prod.bi_ods_worldlinenl.dama_tk_cod
pass_ods_con = spark.table("prod.bi_ods_worldlinenl.dama_tk_con")
contstatus_active = pass_ods_cod.filter(
    (F.col("CODETYPE") == "ContStatus") & (F.col("CODEVAL") == "1")
)
pass_active_contracts = (
    pass_ods_con.alias("con")
    .join(
        contstatus_active.alias("cod"),
        (F.col("con.STATUSVALUE").cast("string") == F.col("cod.CODEVAL")) &
        (F.col("con.InstitutionIDFromFileName") == F.col("cod.InstitutionIdFromFileName")),
        "inner"
    )
    .groupBy(F.col("con.PARTNERID"))
    .agg(F.count("*").alias("active_contracts"))
)

# Step 4: Contract categories for type_of_activity
# Source: prod.bi_dwh_worldlinenl.dimcontract_dama.ContractCategory (active contracts only)
pass_contract_info = (
    pass_dim_ct
    .filter(F.col("ContractStatus") == 1)
    .groupBy("OwnerCompanyBK")
    .agg(
        F.collect_set("ContractCategory").alias("contract_categories"),
        F.array_join(F.collect_set("ContractCategoryDescription"), ", ").alias("contract_category_descriptions"),
    )
)

# Step 5: MCC
# Source: prod.bi_dwh_worldlinenl.dimserviceaccount_dama.MCCode (deterministic min per company)
pass_mcc_info = (
    pass_dim_sa_current
    .filter(F.col("MCCode").isNotNull())
    .groupBy("CompanyBK")
    .agg(F.min("MCCode").alias("mcc"))
)

# Step 6: Legal form + KvK + vintage + sourcecoderef
# Source: prod.bi_ods_worldlinenl.dama_tk_par (LEGALFORM, REGISTRY, CREATEDDATE, SOURCECODEREF)
pass_ods_fields = (
    pass_ods_par_current
    .select(
        F.col("PARTNERID").cast("string").alias("par_partnerid"),
        F.trim(F.col("LEGALFORM")).alias("legal_form"),
        F.trim(F.col("REGISTRY")).alias("kvk_number"),
        F.col("CREATEDDATE").alias("vintage_date"),
        F.col("SOURCECODEREF").alias("sourcecoderef"),
    )
    .withColumn("legal_form", F.when(F.col("legal_form") == "", None).otherwise(F.col("legal_form")))
    .withColumn("kvk_number", F.when(F.col("kvk_number") == "", None).otherwise(F.col("kvk_number")))
)

# Step 7: Assemble PASS base (25 columns)
pass_base_v8 = (
    pass_dim_com_current.alias("com")
    .filter(~F.col("com.CompanyCode").isin(SENTINEL_CODES))
    .join(pass_trx_12m.alias("trx"), F.col("com.CompanyBK") == F.col("trx.CompanyBK"), "left")
    .join(pass_last_trx.alias("lt"), F.col("com.CompanyBK") == F.col("lt.CompanyBK"), "left")
    .join(pass_active_contracts.alias("ac"), F.col("com.CompanyCode") == F.col("ac.PARTNERID").cast("string"), "left")
    .join(pass_contract_info.alias("ct"), F.col("com.CompanyBK") == F.col("ct.OwnerCompanyBK"), "left")
    .join(pass_mcc_info.alias("mcc_t"), F.col("com.CompanyBK") == F.col("mcc_t.CompanyBK"), "left")
    .join(pass_ods_fields.alias("ods"), F.col("com.CompanyCode") == F.col("ods.par_partnerid"), "left")
    .select(
        F.lit("PASS").alias("source"),
        # Source: constructed from com.CompanyCode
        F.concat(F.lit("PASS:"), F.col("com.CompanyCode")).alias("psq_entity_key"),
        # Source: prod.bi_dwh_worldlinenl.dimcompany_dama.CompanyCode
        F.col("com.CompanyCode").alias("id"),
        # Source: prod.bi_ods_worldlinenl.dama_tk_par.SOURCECODEREF
        F.col("ods.sourcecoderef"),
        # Source: prod.bi_dwh_worldlinenl.dimcompany_dama.CompanyName
        F.col("com.CompanyName").alias("name"),
        # Source: prod.bi_dwh_worldlinenl.dimcompany_dama.VATNUMBER
        F.col("com.VATNUMBER").alias("vat_number"),
        # Source: prod.bi_ods_worldlinenl.dama_tk_par.REGISTRY (98.9% fill)
        F.col("ods.kvk_number"),
        # Source: prod.bi_dwh_worldlinenl.dimcompany_dama.CountryBK
        F.col("com.CountryBK").alias("country"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama.DatePK (all-time max via dimserviceaccount_dama)
        F.to_date(F.col("lt.last_trx_datepk").cast("string"), "yyyyMMdd").alias("last_trx_date"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama.DatePK (12m min via dimserviceaccount_dama)
        F.to_date(F.col("trx.first_trx_datepk_12m").cast("string"), "yyyyMMdd").alias("first_trx_date_12m"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama.DatePK (12m max via dimserviceaccount_dama)
        F.to_date(F.col("trx.recent_trx_datepk_12m").cast("string"), "yyyyMMdd").alias("recent_trx_date_12m"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama count(*) over 12m
        F.coalesce(F.col("trx.trx_count_12m"), F.lit(0)).alias("trx_count_12m"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama.AmountValue (sum, 12m, EUR)
        F.coalesce(F.col("trx.total_msv_12m_eur"), F.lit(0).cast("decimal(20,2)")).alias("total_msv_12m_eur"),
        # Source: prod.bi_dwh_worldlinenl.facttransaction_dama.AmountValue (max, 12m)
        F.col("trx.max_trx_amount_12m_eur"),
        # Derived: "Active in last 12 months" if trx_count_12m > 0
        F.when(
            F.col("trx.trx_count_12m").isNotNull() & (F.col("trx.trx_count_12m") > 0),
            "Active in last 12 months"
        ).otherwise("No transactions in last 12 months").alias("activity_detail_status"),
        # Derived: binary Active/Inactive
        F.when(
            F.col("trx.trx_count_12m").isNotNull() & (F.col("trx.trx_count_12m") > 0), "Active"
        ).otherwise("Inactive").alias("merchant_activity_status"),
        # Source: prod.bi_dwh_worldlinenl.dimcontract_dama.ContractCategory (same logic as type_of_activity)
        F.when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in F2F_CATEGORIES])) &
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in NF2F_CATEGORIES])),
            "CP+CNP"
        ).when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in F2F_CATEGORIES])), "CP"
        ).when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in NF2F_CATEGORIES])), "CNP"
        ).otherwise("Unknown").alias("type_of_activity"),
        # Source: prod.bi_dwh_worldlinenl.dimcontract_dama.ContractCategory (F2F/NF2F mapping)
        F.when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in F2F_CATEGORIES])) &
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in NF2F_CATEGORIES])),
            "F2F+NF2F"
        ).when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in F2F_CATEGORIES])), "F2F"
        ).when(
            F.arrays_overlap(F.col("ct.contract_categories"), F.array(*[F.lit(c) for c in NF2F_CATEGORIES])), "NF2F"
        ).otherwise("Unknown").alias("contracting_method"),
        # Source: prod.bi_ods_worldlinenl.dama_tk_par.LEGALFORM
        F.col("ods.legal_form"),
        # Source: prod.bi_dwh_worldlinenl.dimserviceaccount_dama.MCCode (deterministic: min per company)
        F.col("mcc_t.mcc").cast("string").alias("mcc"),
        # Source: prod.bi_ods_worldlinenl.dama_tk_par.CREATEDDATE (partner creation date)
        F.col("ods.vintage_date"),
        # Source: prod.bi_dwh_worldlinenl.dimcompany_dama.SourceStatusDescription
        F.coalesce(F.col("com.SourceStatusDescription"), F.lit("Unknown")).alias("contract_status"),
        # Source: prod.bi_ods_worldlinenl.dama_tk_con + dama_tk_cod (CODETYPE=ContStatus, CODEVAL=1)
        F.coalesce(F.col("ac.active_contracts"), F.lit(0)).alias("active_contracts"),
        # Source: prod.bi_dwh_worldlinenl.dimcontract_dama.ContractCategoryDescription
        F.col("ct.contract_category_descriptions"),
        # Derived: flags missing fields
        F.concat_ws("; ",
            F.when(F.col("com.VATNUMBER").isNull(), F.lit("missing_vat")),
            F.when(F.col("ods.kvk_number").isNull(), F.lit("missing_kvk")),
            F.when(F.col("ods.legal_form").isNull(), F.lit("missing_legal_form")),
            F.when(F.col("mcc_t.mcc").isNull(), F.lit("missing_mcc")),
        ).alias("data_quality_notes"),
    )
)

pass_count = pass_base_v8.count()
print(f"PASS Customer Base v8.1: {pass_count:,} rows")
print(f"Columns ({len(pass_base_v8.columns)}): {pass_base_v8.columns}")

In [ ]:
display(pass_base_v8)

In [ ]:
# ══════════════════════════════════════════════════════════════
# COMBINED PSQ CUSTOMER BASE v8.1
# ══════════════════════════════════════════════════════════════
# UNION ALL: Way4 + PASS with full 25-column spec.
# 25th column: sourcecoderef (PASS = dama_tk_par.SOURCECODEREF, Way4 = NULL)
# ══════════════════════════════════════════════════════════════

# Final column order (25 columns — +1 vs v8.3 original: sourcecoderef)
FINAL_COLUMNS = [
    "source", "psq_entity_key", "id", "sourcecoderef", "name", "vat_number", "kvk_number",
    "country", "last_trx_date", "first_trx_date_12m", "recent_trx_date_12m",
    "trx_count_12m", "total_msv_12m_eur", "max_trx_amount_12m_eur",
    "activity_detail_status", "merchant_activity_status",
    "type_of_activity", "contracting_method", "legal_form", "mcc",
    "vintage_date", "contract_status", "active_contracts",
    "contract_category_descriptions", "data_quality_notes"
]

way4_for_union = way4_base_v8.select(*FINAL_COLUMNS)
pass_for_union = pass_base_v8.select(*FINAL_COLUMNS)

psq_customer_base_v8 = way4_for_union.unionByName(pass_for_union)

total_count = psq_customer_base_v8.count()
_sep = "═" * 70
print(f"\n{_sep}")
print(f"  PSQ COMBINED CUSTOMER BASE v8.1")
print(f"{_sep}")
print(f"  Total entities: {total_count:,}")
print(f"  Way4: {way4_count:,} | PASS: {pass_count:,}")
print(f"  Columns: {len(FINAL_COLUMNS)}")
print(f"\n  Column list:")
for i, col in enumerate(FINAL_COLUMNS, 1):
    print(f"    {i:2d}. {col}")

print(f"\n  --- Activity Status by Source ---")
display(
    psq_customer_base_v8
    .groupBy("source", "merchant_activity_status")
    .agg(F.count("*").alias("count"))
    .orderBy("source", "merchant_activity_status")
)

In [ ]:


print(f"\n  --- Activity Status by Source ---")
display(
    psq_customer_base_v8
    .groupBy("source", "contract_status")
    .agg(F.count("*").alias("count"))
    .orderBy("source", "contract_status")
)

In [ ]:
display(psq_customer_base_v8)

# RICOS Analysis

**Purpose:** Explore RICOS Bronze data relevant to PSQ and analyze match coverage against the customer base.

**RICOS sources:**
- `prod.bronze_ricos_prod.gwgkunde4400` — merchant master (KUNDNR, legal form, MCC, country)
- `prod.bronze_ricos_prod.presult4400` — risk/screening results (RISIKO, RISK_ORIG, STATUS)
- `prod.bronze_ricos_prod.tbbo4400` — UBO/SI relationships (REL_TYPE: BO, SI)

**Filter criteria for PSQ:**
- `KUSY LIKE 'PSQ%'` — PSQ system flag
- `KUNDNR RLIKE '^P_M'` — merchant entities (P_M prefix)
- `HISTBIS LIKE '%9999%'` — current records only (not historical)

**Join key formula:** `concat('P_M', lpad(regexp_replace(id, '[^0-9]', ''), 13, '0'))`

In [ ]:
# ══════════════════════════════════════════════════════════════
# RICOS PSQ-ONLY DATA (filtered to PSQ merchants)
# ══════════════════════════════════════════════════════════════
# Source: prod.bronze_ricos_prod.gwgkunde4400
# Filters:
#   HISTBIS LIKE '%9999%'  = current record (not historical)
#   KUSY LIKE 'PSQ%'       = PSQ system flag
#   KUNDNR RLIKE '^P_M'    = merchant entities (not persons P_P)
#
# This gives us ONLY the RICOS merchants that belong to PSQ.
# ══════════════════════════════════════════════════════════════

# RICOS merchants (P_M = merchant entities)
ricos_psq_merchants = (
    ricos_gwg
    .filter(F.col("HISTBIS").like("%9999%"))   # Current records only
    .filter(F.col("KUSY").like("PSQ%"))          # PSQ system flag
    .filter(F.col("KUNDNR").rlike("^P_M"))       # Merchant entities (not P_P persons)
    .select(
        F.col("KUNDNR").alias("ricos_kundnr"),
        F.trim(F.col("NACHNAME")).alias("ricos_name"),
        F.trim(F.col("KD_SPH_03")).alias("ricos_legal_form"),
        F.trim(F.col("BRANCHE")).alias("ricos_mcc"),
        F.trim(F.col("H_LAND")).alias("ricos_country"),
        F.trim(F.col("KD_SPH_01")).alias("ricos_registry_number"),
        F.trim(F.col("KU_ART")).alias("ricos_customer_type"),
        F.col("KUSY").alias("ricos_system_flag"),
        F.col("KD_SPH_25").alias("ricos_kd_sph_25"),
    )
)

# RICOS persons (P_P = UBO/SI persons linked to PSQ merchants)
ricos_psq_persons = (
    ricos_tbbo
    .filter(F.col("KUNDNR").rlike("^P_M"))       # Parent = PSQ merchant
    .select(
        F.col("KUNDNR").alias("merchant_kundnr"),
        F.col("REL_KUNDNR").alias("person_kundnr"),  # P_P person
        F.col("REL_TYPE").alias("relationship_type"),  # BO or SI
    )
    .distinct()
)

merchant_count = ricos_psq_merchants.count()
person_links = ricos_psq_persons.count()
print(f"RICOS PSQ-Only Data:")
print(f"  Merchants (P_M, current, PSQ flag): {merchant_count:,}")
print(f"  Person links (P_P via tbbo4400):    {person_links:,}")
print(f"\n  Relationship type distribution:")
display(
    ricos_psq_persons
    .groupBy("relationship_type")
    .agg(F.count("*").alias("count"), F.countDistinct("merchant_kundnr").alias("distinct_merchants"))
    .orderBy("relationship_type")
)
print(f"\n  Sample RICOS PSQ merchants:")
display(ricos_psq_merchants.limit(10))

In [ ]:
# ══════════════════════════════════════════════════════════════
# RICOS MERCHANT LOOKUP (normalized)
# ══════════════════════════════════════════════════════════════
# Source: prod.bronze_ricos_prod.gwgkunde4400
# KD_SPH_03 = legal form (96.4% fill, needs case normalization)
# KD_SPH_01 = registry/KvK candidate (numeric, not VAT)
# BRANCHE   = RICOS MCC
# ══════════════════════════════════════════════════════════════

ricos_merchant_lookup = (
    ricos_gwg
    .filter(F.col("HISTBIS").like("%9999%"))
    .filter(F.col("KUSY").like("PSQ%"))
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .select(
        F.col("KUNDNR").alias("ricos_kundnr"),
        F.trim(F.col("NACHNAME")).alias("ricos_name"),
        F.trim(F.col("KD_SPH_03")).alias("ricos_legal_form"),
        # Normalized legal form (title case + known corrections)
        F.when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%BESLOTEN VENNOOTSCHAP%"), F.lit("Besloten Vennootschap (BV)"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%EENMANSZAAK%"), F.lit("Eenmanszaak"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%VENNOOTSCHAP ONDER FIRMA%"), F.lit("Vennootschap onder firma (VOF)"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%COMMANDITAIRE%"), F.lit("Commanditaire Vennootschap (CV)"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%STICHTING%"), F.lit("Stichting"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%VERENIGING%"), F.lit("Vereniging"))
        .when(F.upper(F.trim(F.col("KD_SPH_03"))).like("%PUBLIEKRECHTELIJK%"), F.lit("Publiekrechtelijk"))
        .otherwise(F.trim(F.col("KD_SPH_03"))).alias("ricos_legal_form_normalized"),
        F.trim(F.col("BRANCHE")).alias("ricos_mcc"),
        F.trim(F.col("H_LAND")).alias("ricos_country"),
        F.when(F.trim(F.col("KD_SPH_01")) != "", F.trim(F.col("KD_SPH_01"))).alias("ricos_registry_number"),
        F.trim(F.col("KU_ART")).alias("ricos_customer_type"),
    )
)

print(f"RICOS merchant lookup: {ricos_merchant_lookup.count():,} rows")
display(ricos_merchant_lookup.limit(10))

In [ ]:
# ══════════════════════════════════════════════════════════════
# RICOS RISK LOOKUP (presult4400)
# ══════════════════════════════════════════════════════════════
# Source: prod.bronze_ricos_prod.presult4400
# 100% coverage for RICOS merchants. Deduplicated by HISTVON DESC.
# RISIKO = final composite risk score
# RISK_ORIG = original calculated risk
# RISK_MAN = manual override (if any)
# RISIKOERBE = inherited risk from parent
# PSTATUS + STATUS = screening result
# ══════════════════════════════════════════════════════════════

_risk_window = Window.partitionBy("KUNDNR").orderBy(F.col("HISTVON").desc_nulls_last())

ricos_risk_lookup = (
    ricos_presult
    .filter(F.col("HISTBIS").like("%9999%"))
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .withColumn("_rn", F.row_number().over(_risk_window))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("KUNDNR").alias("risk_kundnr"),
        F.col("RISIKO").alias("risk_final"),
        F.col("RISK_ORIG").alias("risk_original"),
        F.col("RISK_MAN").alias("risk_manual"),
        F.col("RISIKOERBE").alias("risk_inherited"),
        F.concat_ws(" — ", F.col("PSTATUS"), F.col("STATUS")).alias("risk_screening_status"),
    )
)

print(f"RICOS risk lookup: {ricos_risk_lookup.count():,} rows (after dedup)")
print(f"\n  Risk score distribution:")
display(
    ricos_risk_lookup
    .groupBy("risk_final")
    .agg(F.count("*").alias("merchants"))
    .orderBy("risk_final")
)

In [ ]:
# ══════════════════════════════════════════════════════════════
# PSQ-RICOS MATCH RATES
# ══════════════════════════════════════════════════════════════
# Join key: concat('P_M', lpad(regexp_replace(id, '[^0-9]', ''), 13, '0'))
# This transforms the merchant ID to RICOS KUNDNR format.
# Expected: WAY4 ~71.4% match, PASS ~35.3% match
# ══════════════════════════════════════════════════════════════

# Build join key on PSQ customer base
psq_with_key = psq_customer_base_v8.withColumn(
    "ricos_join_key",
    F.concat(
        F.lit("P_M"),
        F.lpad(F.regexp_replace(F.col("id").cast("string"), "[^0-9]", ""), 13, "0")
    )
)

# RICOS current PSQ merchant KUNDNRs (for matching)
ricos_gwg_current = (
    ricos_gwg
    .filter(F.col("HISTBIS").like("%9999%"))
    .filter(F.col("KUSY").like("PSQ%"))
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .select("KUNDNR")
    .distinct()
)

# Match rate by source
psq_match_summary = (
    psq_with_key.alias("cb")
    .join(
        ricos_gwg_current.alias("r"),
        F.col("cb.ricos_join_key") == F.col("r.KUNDNR"),
        "left"
    )
    .withColumn("in_ricos", F.when(F.col("r.KUNDNR").isNotNull(), "Y").otherwise("N"))
    .groupBy("cb.source", "in_ricos")
    .agg(
        F.count("*").alias("merchants"),
        F.sum("cb.total_msv_12m_eur").alias("total_msv_12m_eur"),
        F.sum(F.when(F.col("cb.merchant_activity_status") == "Active", 1).otherwise(0)).alias("active_merchants"),
    )
    .orderBy("source", "in_ricos")
)

print("PSQ-RICOS MATCH RATES")
print("=" * 70)
print(f"  Key formula: concat('P_M', lpad(regexp_replace(id, '[^0-9]', ''), 13, '0'))")
print(f"  RICOS gwgkunde current PSQ merchants: {ricos_gwg_current.count():,}")
print(f"\n  Match breakdown by source:")
display(psq_match_summary)

# Summary percentages
print(f"\n  Match rate summary:")
for src in ["WAY4", "PASS"]:
    total = psq_with_key.filter(F.col("source") == src).count()
    matched = psq_with_key.alias("cb").join(
        ricos_gwg_current.alias("r"),
        F.col("cb.ricos_join_key") == F.col("r.KUNDNR"), "inner"
    ).filter(F.col("cb.source") == src).count()
    pct = 100 * matched / total if total > 0 else 0
    print(f"    {src}: {matched:,} / {total:,} = {pct:.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════
# UNMATCHED ROOT CAUSE ANALYSIS
# ══════════════════════════════════════════════════════════════
# WHY are unmatched merchants NOT in RICOS?
#
# Root causes (from PSQ PASS/WAY4 Unmatched Investigation notebooks):
#   WAY4: Merchants onboarded AFTER the RICOS initial load (2022-03-29)
#         were never registered. Timing gap, not deliberate exclusion.
#   PASS: Merchants that existed BEFORE the RICOS load were deliberately
#         excluded from registration. Opposite pattern to WAY4.
#
# Both are operational gaps — not data bugs.
# ══════════════════════════════════════════════════════════════

# Identify unmatched merchants
psq_unmatched = (
    psq_with_key.alias("cb")
    .join(
        ricos_gwg_current.alias("r"),
        F.col("cb.ricos_join_key") == F.col("r.KUNDNR"),
        "left_anti"
    )
)

print("UNMATCHED ROOT CAUSE ANALYSIS")
print("=" * 70)

# Breakdown by source + activity + contract status
print(f"\n  Total unmatched: {psq_unmatched.count():,}")
print(f"\n  By source + activity status:")
display(
    psq_unmatched
    .groupBy("source", "merchant_activity_status")
    .agg(
        F.count("*").alias("merchants"),
        F.sum("total_msv_12m_eur").alias("total_msv_12m_eur"),
    )
    .orderBy("source", "merchant_activity_status")
)

print(f"\n  By source + contract status (shows WHY they're unmatched):")
display(
    psq_unmatched
    .groupBy("source", "contract_status")
    .agg(
        F.count("*").alias("merchants"),
        F.sum(F.when(F.col("merchant_activity_status") == "Active", 1).otherwise(0)).alias("active_count"),
        F.sum("total_msv_12m_eur").alias("total_msv_12m_eur"),
    )
    .orderBy("source", "contract_status")
)

# WAY4 unmatched: check vintage dates (onboarded after RICOS load 2022-03-29)
print(f"\n  WAY4 unmatched — vintage date distribution (RICOS load = 2022-03-29):")
display(
    psq_unmatched
    .filter(F.col("source") == "WAY4")
    .withColumn("onboarded_after_ricos",
        F.when(F.col("vintage_date") > "2022-03-29", "Yes").otherwise("No / Unknown")
    )
    .groupBy("onboarded_after_ricos")
    .agg(F.count("*").alias("merchants"), F.sum("total_msv_12m_eur").alias("msv_eur"))
    .orderBy("onboarded_after_ricos")
)

# PASS unmatched: check contract status (mostly Cancelled = expected)
print(f"\n  PASS unmatched — contract status breakdown:")
display(
    psq_unmatched
    .filter(F.col("source") == "PASS")
    .groupBy("contract_status")
    .agg(
        F.count("*").alias("merchants"),
        F.sum(F.when(F.col("merchant_activity_status") == "Active", 1).otherwise(0)).alias("active"),
        F.sum("total_msv_12m_eur").alias("msv_eur"),
    )
    .orderBy(F.desc("merchants"))
)

print(f"\n  Root cause summary:")
print(f"    WAY4: Onboarded after RICOS initial load (2022-03-29) → never registered")
print(f"    PASS: Existed before RICOS load → deliberately excluded from registration")
print(f"    Both = operational gap, not data bug")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PSQ CUSTOMER BASE + in_ricos_flag (Y/N)
# ══════════════════════════════════════════════════════════════
# Full PSQ base (25 columns) + 1 additional column: in_ricos_flag
# Y = merchant KUNDNR found in gwgkunde4400 (current PSQ records)
# N = no match in RICOS
#
# This is the final output DataFrame for downstream analysis.
# ══════════════════════════════════════════════════════════════

psq_with_ricos_flag = (
    psq_with_key.alias("cb")
    .join(
        ricos_gwg_current.alias("r"),
        F.col("cb.ricos_join_key") == F.col("r.KUNDNR"),
        "left"
    )
    .select(
        *[F.col(f"cb.{c}") for c in psq_customer_base_v8.columns],
        F.when(F.col("r.KUNDNR").isNotNull(), "Y").otherwise("N").alias("in_ricos_flag"),
    )
)

_total = psq_with_ricos_flag.count()
_in_ricos = psq_with_ricos_flag.filter(F.col("in_ricos_flag") == "Y").count()
_not_in_ricos = _total - _in_ricos

print(f"PSQ Customer Base with in_ricos_flag")
print("=" * 70)
print(f"  Total merchants:  {_total:,}")
print(f"  In RICOS (Y):     {_in_ricos:,} ({100*_in_ricos/_total:.1f}%)")
print(f"  Not in RICOS (N): {_not_in_ricos:,} ({100*_not_in_ricos/_total:.1f}%)")
print(f"  Columns: {len(psq_with_ricos_flag.columns)} (25 base + in_ricos_flag)")
print(f"\n  Coverage by source + activity:")
display(
    psq_with_ricos_flag
    .groupBy("source", "merchant_activity_status", "in_ricos_flag")
    .agg(F.count("*").alias("merchants"), F.sum("total_msv_12m_eur").alias("msv_12m_eur"))
    .orderBy("source", "merchant_activity_status", "in_ricos_flag")
)

display(
    psq_with_ricos_flag
    .groupBy("source", "contract_status", "in_ricos_flag")
    .agg(F.count("*").alias("merchants"), F.sum("total_msv_12m_eur").alias("msv_12m_eur"))
    .orderBy("source", "contract_status", "in_ricos_flag")
)
print(f"\n  Full DataFrame:")
display(psq_with_ricos_flag)


In [ ]:
# ══════════════════════════════════════════════════════════════
# WRITE psq_with_ricos_flag TO TABLE
# ══════════════════════════════════════════════════════════════
# Target: projects.riskdatascience.psq_aml_with_ricosflag
# Note: Saved as a managed table (not a view) because the DataFrame
# is built from complex PySpark joins that reference temp objects.
# ══════════════════════════════════════════════════════════════

(
    psq_with_ricos_flag
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("projects.riskdatascience.psq_aml_with_ricosflag")
)

print("✓ Written to projects.riskdatascience.psq_aml_with_ricosflag")
print(f"  Rows: {spark.table('projects.riskdatascience.psq_aml_with_ricosflag').count():,}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PSQ WITH RICOS RICH (psq_with_ricos_flag + 25 RICOS columns)
# ══════════════════════════════════════════════════════════════
# Enriches psq_with_ricos_flag with:
#   gwgkunde4400: 10 identity columns
#   presult4400:  13 risk & evidence columns
#   tbbo4400:      2 UBO/SI count columns
#
# Merchants with in_ricos_flag='N' get NULL for all enrichment
# columns (natural behavior of left join on ricos_join_key).
#
# Final shape: 51 columns (26 base + 25 RICOS enrichment)
# ══════════════════════════════════════════════════════════════

from pyspark.sql.window import Window

# ── Step 1: Add join key to psq_with_ricos_flag ──
psq_base = psq_with_ricos_flag.withColumn(
    "ricos_join_key",
    F.concat(
        F.lit("P_M"),
        F.lpad(F.regexp_replace(F.col("id").cast("string"), "[^0-9]", ""), 13, "0")
    )
)

# ── Step 2: gwgkunde4400 enrichment (10 identity columns) ──
ricos_gwg_enrichment = (
    spark.table("prod.bronze_ricos_prod.gwgkunde4400")
    .filter(F.col("HISTBIS").like("%9999%"))
    .filter(F.col("KUSY").like("PSQ%"))
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .select(
        F.col("KUNDNR").alias("_gwg_key"),
        F.trim(F.col("NACHNAME")).alias("ricos_name"),
        F.trim(F.col("STRASSE")).alias("ricos_street"),
        F.trim(F.col("PLZ")).alias("ricos_postal_code"),
        F.trim(F.col("WOHNORT")).alias("ricos_city"),
        F.trim(F.col("BRANCHE")).alias("ricos_mcc"),
        F.trim(F.col("H_LAND")).alias("ricos_country"),
        F.trim(F.col("KD_SPH_03")).alias("ricos_legal_form"),
        F.when(F.trim(F.col("KD_SPH_06")) != "", F.trim(F.col("KD_SPH_06"))).alias("ricos_kvk"),
        F.when(F.trim(F.col("KD_SPH_01")) != "", F.trim(F.col("KD_SPH_01"))).alias("ricos_duns_id"),
        F.when(F.trim(F.col("KD_SPH_05")) != "", F.trim(F.col("KD_SPH_05"))).alias("ricos_vat"),
    )
)

# ── Step 3: presult4400 enrichment (13 risk & evidence columns) ──
# Deduplicated: one row per KUNDNR, most recent HISTVON wins
_risk_window = Window.partitionBy("KUNDNR").orderBy(F.col("HISTVON").desc_nulls_last())

ricos_presult_enrichment = (
    spark.table("prod.bronze_ricos_prod.presult4400")
    .filter(F.col("HISTBIS").like("%9999%"))
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .withColumn("_rn", F.row_number().over(_risk_window))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("KUNDNR").alias("_presult_key"),
        F.trim(F.col("RISIKO")).alias("ricos_risk_score"),
        F.trim(F.col("RIK_BEZ")).alias("ricos_risk_label"),
        F.trim(F.col("RISK_ORIG_NAME")).alias("ricos_risk_original"),
        F.when(F.trim(F.col("RISK_MAN_NAME")) != "", F.trim(F.col("RISK_MAN_NAME"))).alias("ricos_risk_manual"),
        F.when(F.trim(F.col("RISK_INHERIT_NAME")) != "", F.trim(F.col("RISK_INHERIT_NAME"))).alias("ricos_risk_inherited"),
        F.trim(F.col("STATUS")).alias("ricos_screening_status"),
        F.trim(F.col("PSTATUS")).alias("ricos_screening_pstatus"),
        F.when(F.col("TREFFERPROZ") != "0", F.col("TREFFERPROZ")).alias("ricos_watchlist_hit_pct"),
        F.when(F.trim(F.col("SL_LISTNAME")) != "", F.trim(F.col("SL_LISTNAME"))).alias("ricos_watchlist_list"),
        F.when(F.col("HITVAL_EMB") != "0", F.col("HITVAL_EMB")).alias("ricos_embargo_hit_pct"),
        F.when(F.col("HITVAL_PEP") != "0", F.col("HITVAL_PEP")).alias("ricos_pep_hit_pct"),
        F.when(F.trim(F.col("REMINDDATE")) != "", F.to_date(F.col("REMINDDATE"), "yyyyMMdd")).alias("ricos_next_review_date"),
        F.when(F.trim(F.col("KOMMENTAR")) != "", F.trim(F.col("KOMMENTAR"))).alias("ricos_risk_comment"),
    )
)

# ── Step 4: tbbo4400 enrichment (2 UBO/SI count columns) ──
ricos_tbbo_enrichment = (
    spark.table("prod.bronze_ricos_prod.tbbo4400")
    .filter(F.col("KUNDNR").rlike("^P_M"))
    .filter(F.col("HISTBIS").like("%9999%"))
    .groupBy("KUNDNR")
    .agg(
        F.sum(F.when(F.trim(F.col("REL_TYPE")) == "BO", 1).otherwise(0)).alias("ricos_ubo_count"),
        F.sum(F.when(F.trim(F.col("REL_TYPE")) == "SI", 1).otherwise(0)).alias("ricos_si_count"),
    )
    .withColumnRenamed("KUNDNR", "_tbbo_key")
)

# ── Step 5: Left join all enrichment onto psq_with_ricos_flag ──
psq_with_ricos_rich = (
    psq_base
    .join(ricos_gwg_enrichment, psq_base["ricos_join_key"] == ricos_gwg_enrichment["_gwg_key"], "left")
    .join(ricos_presult_enrichment, psq_base["ricos_join_key"] == ricos_presult_enrichment["_presult_key"], "left")
    .join(ricos_tbbo_enrichment, psq_base["ricos_join_key"] == ricos_tbbo_enrichment["_tbbo_key"], "left")
    .drop("ricos_join_key", "_gwg_key", "_presult_key", "_tbbo_key")
)

# ── Summary ──
_total = psq_with_ricos_rich.count()
_enriched = psq_with_ricos_rich.filter(F.col("ricos_risk_score").isNotNull()).count()
_null = _total - _enriched

print(f"PSQ WITH RICOS RICH")
print("=" * 70)
print(f"  Total merchants:   {_total:,}")
print(f"  RICOS enriched:    {_enriched:,} ({100*_enriched/_total:.1f}%)")
print(f"  RICOS NULL (N):    {_null:,} ({100*_null/_total:.1f}%)")
print(f"  Total columns:     {len(psq_with_ricos_rich.columns)}")
print(f"    Base (PSQ):      26 (25 + in_ricos_flag)")
print(f"    gwgkunde4400:    10 (identity)")
print(f"    presult4400:     13 (risk & evidence)")
print(f"    tbbo4400:         2 (UBO/SI counts)")
print(f"\n  Enrichment columns:")
ricos_cols = [c for c in psq_with_ricos_rich.columns if c.startswith("ricos_")]
for i, c in enumerate(ricos_cols, 1):
    print(f"    {i:2d}. {c}")

print(f"\n  Risk distribution (enriched merchants):")
display(
    psq_with_ricos_rich
    .filter(F.col("ricos_risk_label").isNotNull())
    .groupBy("ricos_risk_label")
    .agg(F.count("*").alias("merchants"))
    .orderBy(F.desc("merchants"))
)

print(f"\n  Watchlist hits (non-zero TREFFERPROZ):")
_wl_hits = psq_with_ricos_rich.filter(F.col("ricos_watchlist_hit_pct").isNotNull()).count()
print(f"    {_wl_hits:,} merchants with active watchlist match")

print(f"\n  KYC review overdue:")
_overdue = psq_with_ricos_rich.filter(F.col("ricos_next_review_date") < F.current_date()).count()
print(f"    {_overdue:,} merchants with overdue review date")

display(psq_with_ricos_rich)